In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd

In [2]:
df_train1 = pd.read_csv(
    "C:/Users/goura/Sem5/Machine Learning/BT2024213/BT2024213/BT2024213_train_var1.csv"
)
display(df_train1.head());

,x1,x2,x3,x4,x5,x6,y
0,-1.000000,0.520692,-0.406060,-0.836003,0.150516,0.472777,0.398249
1,0.741769,-1.000000,0.273458,-0.300593,-0.624172,-0.167396,2.219173
2,-0.753915,-0.252239,-1.000000,-0.406372,0.304597,0.916086,5.645412
3,1.000000,1.000000,0.612335,1.000000,1.000000,-0.221625,-4.180678
4,-1.000000,-1.000000,-0.274695,0.396140,0.867543,0.865820,2.658020


In [3]:
df_test1 = pd.read_csv(
    "C:/Users/goura/Sem5/Machine Learning/BT2024213/BT2024213/BT2024213_test_var1.csv"
)
display(df_test1.head());

,x1,x2,x3,x4,x5,x6
0,1.000000,0.468421,1.000000,1.000000,1.000000,-1.000000
1,-0.478846,-0.775865,-0.426413,-1.000000,0.232556,-0.288742
2,-1.000000,1.000000,-1.000000,0.832330,-0.503098,-1.000000
3,1.000000,0.903710,1.000000,0.948196,1.000000,-0.194109
4,1.000000,1.000000,0.625929,0.780418,0.933867,-1.000000


N = df_train1.shape[0]
ones = np.ones((N,1))
X_raw = df_train1.iloc[:, :6].values
X = np.hstack((ones, X_raw));
print(X[0])
print(X.shape)


In [4]:
Y = df_train1.iloc[:, 6:].values
print(Y.shape)
print(Y[0])

(1000, 1)
[0.3982494]


In [5]:
N = df_train1.shape[0]
ones = np.ones((N,1))
X_raw = df_train1.iloc[:, :6].values
X = np.hstack((ones, X_raw));
print(X[0])
print(X.shape)

[ 1.         -1.          0.52069213 -0.40606021 -0.8360028   0.1505158
  0.47277675]
(1000, 7)


In [8]:
Y_raw = df_train1.iloc[:, -1].values

In [18]:
import torch

print("PyTorch:", torch.__version__)
print("CUDA compiled into PyTorch:", torch.version.cuda)
print("CUDA available:", torch.cuda.is_available())



PyTorch: 2.11.0+cu128
CUDA compiled into PyTorch: 12.8
CUDA available: True


### Fast Elastic Net via PyTorch Autograd + Adam

By formulating Elastic Net as a loss minimization problem and optimizing all parameters concurrently using PyTorch's `Adam` optimizer, we leverage the parallel processing power of the GPU to complete the hyperparameter grid search rapidly.

### Enhanced GPU Polynomial Feature Generator with Interaction Terms

We will redefine `get_polynomial_features_torch` to generate all interactions and cross-feature combinations up to the chosen degree using combinations with replacement.

In [11]:
import torch
import torch.optim as optim
import numpy as np
import time
import gc
from itertools import combinations_with_replacement

# Ensure we use GPU if available
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

def get_polynomial_features_torch_with_interactions(X_raw, degree):
    N, D = X_raw.shape
    # Start with bias (ones)
    features = [torch.ones((N, 1), dtype=torch.float64, device=device)]

    # Generate all combinations of features from degree 1 up to 'degree'
    for d in range(1, degree + 1):
        for combos in combinations_with_replacement(range(D), d):
            # Compute product of selected features for this combination
            prod = torch.prod(X_raw[:, combos], dim=1, keepdim=True)
            features.append(prod)

    return torch.hstack(features)

X_raw_torch = torch.tensor(X_raw, dtype=torch.float64, device=device)
Y_torch = torch.tensor(Y_raw, dtype=torch.float64, device=device).reshape(-1, 1)

test_feat = get_polynomial_features_torch_with_interactions(X_raw_torch, 3)
print(f"New feature set shape for degree 3 (including interactions): {test_feat.shape}")

Using device: cuda
New feature set shape for degree 3 (including interactions): torch.Size([1000, 84])


In [12]:
print("Device:", device)
print("X_raw:", X_raw_torch.device)


Device: cuda
X_raw: cuda:0


In [13]:
def elastic_net_adam_torch(X, Y, l1_ratio=0.5, alpha=0.01, lr=0.05, epochs=2000):
    N, D = X.shape
    beta = torch.zeros((D, 1), dtype=torch.float64, device=X.device, requires_grad=True)
    optimizer = torch.optim.Adam([beta], lr=lr)

    for epoch in range(epochs):
        optimizer.zero_grad()
        pred = X @ beta
        mse = torch.mean((pred - Y) ** 2)
        # Penalty excludes the bias term (first column of ones)
        l1 = torch.sum(torch.abs(beta[1:]))
        l2 = torch.sum(beta[1:] ** 2)
        loss = mse + alpha * (l1_ratio * l1 + 0.5 * (1 - l1_ratio) * l2)
        loss.backward()
        optimizer.step()

    return beta.detach()

In [15]:
# Rerun grid search using new comprehensive interaction feature sets
alphas = [0.005, 0.01, 0.05]
l1_ratios = [0.1, 0.3, 0.5, 0.7, 0.9]
degrees = [2, 3, 4, 5, 6,7] # Higher degrees with interactions will produce large numbers of features
K_folds = 10

best_mse = float('inf')
best_params = {}

start_time = time.time()
print("Starting parallelized PyTorch autograd grid search with interactions...\n")

for deg in degrees:
    X_poly = get_polynomial_features_torch_with_interactions(X_raw_torch, deg)

    # Split setup
    torch.manual_seed(42)
    indices = torch.randperm(X_poly.size(0), device=device)
    X_poly_shuffled = X_poly[indices]
    Y_shuffled = Y_torch[indices]

    X_folds = torch.chunk(X_poly_shuffled, K_folds, dim=0)
    Y_folds = torch.chunk(Y_shuffled, K_folds, dim=0)

    for alpha in alphas:
        for l1_ratio in l1_ratios:
            validation_mses = []

            for i in range(K_folds):
                X_val, Y_val = X_folds[i], Y_folds[i]

                X_train = torch.cat([X_folds[j] for j in range(K_folds) if j != i], dim=0)
                Y_train = torch.cat([Y_folds[j] for j in range(K_folds) if j != i], dim=0)

                beta_en = elastic_net_adam_torch(
                    X_train, Y_train, l1_ratio=l1_ratio, alpha=alpha, lr=0.05, epochs= 500
                )

                Y_val_pred = X_val.matmul(beta_en)
                fold_mse = torch.mean((Y_val - Y_val_pred) ** 2).item()
                validation_mses.append(fold_mse)

            mean_mse = np.mean(validation_mses)
            if mean_mse < best_mse:
                best_mse = mean_mse
                best_params = {'degree': deg, 'alpha': alpha, 'l1_ratio': l1_ratio}
                print(f"* New Best! Degree: {deg}, Alpha: {alpha}, L1 Ratio: {l1_ratio} -> Mean CV MSE: {mean_mse:.5f}")

    del X_poly
    gc.collect()

    if torch.cuda.is_available():
        torch.cuda.empty_cache()

end_time = time.time()
print(f"\nGrid Search completed in {end_time - start_time:.2f} seconds!")
print(f"Optimal Parameters: {best_params}")
print(f"Lowest Validation MSE achieved: {best_mse:.5f}")

Starting parallelized PyTorch autograd grid search with interactions...

* New Best! Degree: 2, Alpha: 0.005, L1 Ratio: 0.1 -> Mean CV MSE: 2.95363
* New Best! Degree: 2, Alpha: 0.005, L1 Ratio: 0.3 -> Mean CV MSE: 2.95276
* New Best! Degree: 2, Alpha: 0.005, L1 Ratio: 0.5 -> Mean CV MSE: 2.95191
* New Best! Degree: 2, Alpha: 0.005, L1 Ratio: 0.7 -> Mean CV MSE: 2.95105
* New Best! Degree: 2, Alpha: 0.005, L1 Ratio: 0.9 -> Mean CV MSE: 2.95005
* New Best! Degree: 2, Alpha: 0.01, L1 Ratio: 0.5 -> Mean CV MSE: 2.94910
* New Best! Degree: 2, Alpha: 0.01, L1 Ratio: 0.7 -> Mean CV MSE: 2.94743
* New Best! Degree: 2, Alpha: 0.01, L1 Ratio: 0.9 -> Mean CV MSE: 2.94606
* New Best! Degree: 3, Alpha: 0.005, L1 Ratio: 0.1 -> Mean CV MSE: 1.02714
* New Best! Degree: 3, Alpha: 0.005, L1 Ratio: 0.3 -> Mean CV MSE: 1.02146
* New Best! Degree: 3, Alpha: 0.005, L1 Ratio: 0.5 -> Mean CV MSE: 1.01708
* New Best! Degree: 3, Alpha: 0.005, L1 Ratio: 0.7 -> Mean CV MSE: 1.01345
* New Best! Degree: 3, Alpha: 

In [ ]:
best_degree = best_params.get('degree', 3)
best_alpha = best_params.get('alpha', 0.01)
best_l1_ratio = best_params.get('l1_ratio', 0.9)

X_poly_full = get_polynomial_features_torch_with_interactions(X_raw_torch, best_degree)
final_beta = elastic_net_adam_torch(
    X_poly_full, Y_torch, l1_ratio=best_l1_ratio, alpha=best_alpha, lr=0.05, epochs=2000
)

print("\nFinal Model Weights (first 10 terms):")
print(final_beta.flatten()[:10])

In [17]:
import pandas as pd

# --- Grid: pushed toward smaller alpha / higher L1, since the old best was on the edge ---
alphas = [0.0005, 0.001, 0.002, 0.005]
l1_ratios = [0.5, 0.7, 0.9, 0.95, 0.99, 1.0]   # 0.1 and 0.3 were consistently worse
degrees = [4, 5, 6, 7]
K_folds = 10

# More epochs for higher degrees so they get a fair chance to converge
epochs_by_degree = {2: 500, 3: 500, 4: 500, 5: 750, 6: 1000, 7: 1500}
lr = 0.05

results = []
best_mse = float('inf')
best_params = {}

start_time = time.time()
print("Starting PyTorch autograd grid search with interactions...\n")

for deg in degrees:
    deg_start = time.time()
    X_poly = get_polynomial_features_torch_with_interactions(X_raw_torch, deg)
    n_features = X_poly.size(1)
    epochs = epochs_by_degree[deg]
    print(f"--- Degree {deg}: {n_features} features, {epochs} epochs ---")

    # Same shuffle for every degree so folds are comparable
    torch.manual_seed(42)
    indices = torch.randperm(X_poly.size(0), device=device)
    X_folds = torch.chunk(X_poly[indices], K_folds, dim=0)
    Y_folds = torch.chunk(Y_torch[indices], K_folds, dim=0)

    for alpha in alphas:
        for l1_ratio in l1_ratios:
            val_mses, train_mses = [], []

            for i in range(K_folds):
                X_val, Y_val = X_folds[i], Y_folds[i]
                X_train = torch.cat([X_folds[j] for j in range(K_folds) if j != i], dim=0)
                Y_train = torch.cat([Y_folds[j] for j in range(K_folds) if j != i], dim=0)

                beta_en = elastic_net_adam_torch(
                    X_train, Y_train, l1_ratio=l1_ratio, alpha=alpha, lr=lr, epochs=epochs
                )

                with torch.no_grad():
                    val_mses.append(torch.mean((Y_val - X_val @ beta_en) ** 2).item())
                    train_mses.append(torch.mean((Y_train - X_train @ beta_en) ** 2).item())

            mean_val = np.mean(val_mses)
            mean_train = np.mean(train_mses)
            results.append({
                'degree': deg, 'alpha': alpha, 'l1_ratio': l1_ratio,
                'n_features': n_features, 'epochs': epochs,
                'val_mse': mean_val, 'val_std': np.std(val_mses),
                'train_mse': mean_train,
            })

            marker = ""
            if mean_val < best_mse:
                best_mse = mean_val
                best_params = {'degree': deg, 'alpha': alpha, 'l1_ratio': l1_ratio}
                marker = "  * New Best!"

            print(f"Deg {deg} | alpha {alpha:<7} | L1 {l1_ratio:<5} -> "
                  f"val {mean_val:.5f} (±{np.std(val_mses):.4f}) | train {mean_train:.5f}{marker}")

    print(f"Degree {deg} done in {time.time() - deg_start:.1f}s\n")

    del X_poly, X_folds, Y_folds
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()

print(f"Grid Search completed in {time.time() - start_time:.2f} seconds!")
print(f"Optimal Parameters: {best_params}")
print(f"Lowest Validation MSE achieved: {best_mse:.5f}")

# --- Summary tables ---
df = pd.DataFrame(results)

print("\nValidation MSE (rows: degree, alpha | cols: l1_ratio)")
print(df.pivot_table(index=['degree', 'alpha'], columns='l1_ratio', values='val_mse').round(4))

print("\nBest config per degree (train vs val tells you overfit vs underfit):")
best_per_deg = df.loc[df.groupby('degree')['val_mse'].idxmin()]
best_per_deg['gap'] = best_per_deg['val_mse'] - best_per_deg['train_mse']
print(best_per_deg[['degree', 'alpha', 'l1_ratio', 'n_features',
                    'train_mse', 'val_mse', 'gap']].round(5).to_string(index=False))



Starting PyTorch autograd grid search with interactions...

--- Degree 4: 210 features, 500 epochs ---
Deg 4 | alpha 0.0005  | L1 0.5   -> val 0.63354 (±0.0847) | train 0.35430  * New Best!
Deg 4 | alpha 0.0005  | L1 0.7   -> val 0.62998 (±0.0840) | train 0.35452  * New Best!
Deg 4 | alpha 0.0005  | L1 0.9   -> val 0.62675 (±0.0833) | train 0.35480  * New Best!
Deg 4 | alpha 0.0005  | L1 0.95  -> val 0.62603 (±0.0832) | train 0.35488  * New Best!
Deg 4 | alpha 0.0005  | L1 0.99  -> val 0.62543 (±0.0830) | train 0.35495  * New Best!
Deg 4 | alpha 0.0005  | L1 1.0   -> val 0.62528 (±0.0830) | train 0.35496  * New Best!
Deg 4 | alpha 0.001   | L1 0.5   -> val 0.61861 (±0.0789) | train 0.35605  * New Best!
Deg 4 | alpha 0.001   | L1 0.7   -> val 0.61435 (±0.0777) | train 0.35680  * New Best!
Deg 4 | alpha 0.001   | L1 0.9   -> val 0.61077 (±0.0767) | train 0.35770  * New Best!
Deg 4 | alpha 0.001   | L1 0.95  -> val 0.60976 (±0.0765) | train 0.35794  * New Best!
Deg 4 | alpha 0.001   | L1 

In [20]:
import numpy as np
import pandas as pd
import warnings
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import ElasticNetCV
from sklearn.model_selection import KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures, StandardScaler

X = X_raw_torch.cpu().numpy()
y = Y_torch.cpu().numpy().ravel()
kf = KFold(n_splits=10, shuffle=True, random_state=42)

rows = []
for deg in range(2, 8):
    pipe = make_pipeline(
        MinMaxScaler(feature_range=(-1, 1)),            # keeps x^deg from blowing up
        PolynomialFeatures(degree=deg, include_bias=False),
        StandardScaler(),                                # equal footing for the penalty
        ElasticNetCV(l1_ratio=[0.5, 0.7, 0.9, 0.95, 0.99, 1.0],
                     alphas=60, cv=kf, max_iter=50000, n_jobs=-1),
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        pipe.fit(X, y)

    enet = pipe[-1]
    cv_mse = enet.mse_path_.mean(axis=-1).min()   # mean over folds, best (l1, alpha)
    rows.append({'degree': deg,
                 'n_features': enet.coef_.size,
                 'nonzero': int(np.sum(enet.coef_ != 0)),
                 'alpha': enet.alpha_, 'l1_ratio': enet.l1_ratio_,
                 'cv_mse': cv_mse})
    print(f"Degree {deg:>2}: CV MSE {cv_mse:.4f} | alpha {enet.alpha_:.2e} | "
          f"L1 {enet.l1_ratio_} | {rows[-1]['nonzero']}/{enet.coef_.size} nonzero")

df = pd.DataFrame(rows)
best = df.loc[df['cv_mse'].idxmin()]
print(f"\nBest degree: {int(best['degree'])}, CV MSE {best['cv_mse']:.4f}, "
      f"R² {1 - best['cv_mse'] / np.var(y):.4f}")

Degree  2: CV MSE 2.9372 | alpha 1.74e-02 | L1 1.0 | 22/27 nonzero
Degree  3: CV MSE 1.0097 | alpha 5.39e-03 | L1 1.0 | 71/83 nonzero
Degree  4: CV MSE 0.5607 | alpha 8.60e-03 | L1 1.0 | 114/209 nonzero
Degree  5: CV MSE 0.3198 | alpha 7.65e-03 | L1 1.0 | 141/461 nonzero
Degree  6: CV MSE 0.3319 | alpha 9.67e-03 | L1 1.0 | 141/923 nonzero


KeyboardInterrupt: 

In [21]:
import time
import warnings
from math import comb

import numpy as np
import pandas as pd
from sklearn.exceptions import ConvergenceWarning
from sklearn.linear_model import LassoCV
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.model_selection import KFold, train_test_split
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import MinMaxScaler, PolynomialFeatures, StandardScaler

# ---------------- Data ----------------
X = X_raw_torch.cpu().numpy()
y = Y_torch.cpu().numpy().ravel()

TEST_SIZE = 0.2          # held-out test set for the final score; set to 0 to use all data for CV
if TEST_SIZE > 0:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=TEST_SIZE, random_state=42)
else:
    X_tr, y_tr = X, y

n_inputs = X.shape[1]
degrees = range(2, 8)    # with 6 inputs, degree 8 = 3,003 features
kf = KFold(n_splits=10, shuffle=True, random_state=42)

# ---------------- Degree sweep ----------------
rows, pipes = [], {}
start = time.time()

for deg in degrees:
    n_feat = comb(n_inputs + deg, deg) - 1
    mem_gb = X_tr.shape[0] * n_feat * 8 / 1e9
    if mem_gb > 4:
        print(f"Skipping degree {deg}: feature matrix would need ~{mem_gb:.1f} GB")
        continue

    t = time.time()
    pipe = make_pipeline(
        MinMaxScaler(feature_range=(-1, 1)),             # keeps high powers bounded
        PolynomialFeatures(degree=deg, include_bias=False),
        StandardScaler(),                                # equal footing for the L1 penalty
        LassoCV(alphas=60, cv=kf, max_iter=50000, n_jobs=-1),
    )
    with warnings.catch_warnings():
        warnings.simplefilter("ignore", ConvergenceWarning)
        pipe.fit(X_tr, y_tr)

    lasso = pipe[-1]
    mean_path = lasso.mse_path_.mean(axis=1)            # mse_path_: (n_alphas, n_folds)
    j = mean_path.argmin()
    fold_mse = lasso.mse_path_[j]
    se = fold_mse.std(ddof=1) / np.sqrt(len(fold_mse))
    at_edge = j in (0, len(mean_path) - 1)

    pipes[deg] = pipe
    rows.append({
        'degree': deg,
        'n_features': lasso.coef_.size,
        'nonzero': int(np.sum(lasso.coef_ != 0)),
        'alpha': lasso.alpha_,
        'cv_mse': fold_mse.mean(),
        'se': se,
        'alpha_at_edge': at_edge,
    })
    print(f"Degree {deg}: CV MSE {fold_mse.mean():.4f} (±{se:.4f}) | alpha {lasso.alpha_:.2e}"
          f"{' [edge of path]' if at_edge else ''} | "
          f"{rows[-1]['nonzero']}/{lasso.coef_.size} nonzero | {time.time() - t:.1f}s")

print(f"\nSweep finished in {time.time() - start:.1f}s")

# ---------------- Pick a degree ----------------
df = pd.DataFrame(rows)
best = df.loc[df['cv_mse'].idxmin()]
chosen = df[df['cv_mse'] <= best['cv_mse'] + best['se']].iloc[0]   # one-standard-error rule

print("\n" + df[['degree', 'nonzero', 'alpha', 'cv_mse', 'se']].round(5).to_string(index=False))
print(f"\nLowest CV MSE : degree {int(best['degree'])} -> {best['cv_mse']:.4f} ± {best['se']:.4f}")
print(f"1-SE choice   : degree {int(chosen['degree'])} -> {chosen['cv_mse']:.4f}")
print(f"CV R² (chosen): {1 - chosen['cv_mse'] / np.var(y_tr):.4f}")

final_deg = int(chosen['degree'])
final_model = pipes[final_deg]          # LassoCV already refit on all training data

# ---------------- Held-out test score ----------------
if TEST_SIZE > 0:
    for label, deg in [('1-SE choice', final_deg), ('Lowest CV', int(best['degree']))]:
        pred = pipes[deg].predict(X_te)
        print(f"Test ({label}, degree {deg}): MSE {mean_squared_error(y_te, pred):.4f} | "
              f"R² {r2_score(y_te, pred):.4f}")

# ---------------- Which terms survived ----------------
names = final_model[1].get_feature_names_out([f"x{i}" for i in range(n_inputs)])
coefs = pd.Series(final_model[-1].coef_, index=names)
kept = coefs[coefs != 0].sort_values(key=np.abs, ascending=False)
print(f"\nDegree {final_deg}: {len(kept)} of {len(coefs)} terms kept. Largest (standardized scale):")
print(kept.head(15).round(4).to_string())

# ---------------- Prediction helper ----------------
X_min, X_max = X_tr.min(axis=0), X_tr.max(axis=0)

def predict(X_new):
    X_new = np.asarray(X_new, dtype=float)
    outside = (X_new < X_min) | (X_new > X_max)
    if outside.any():
        print(f"Warning: {outside.any(axis=1).sum()} row(s) outside the training range; "
              "high-degree predictions there are unreliable.")
    return final_model.predict(X_new)

Degree 2: CV MSE 2.9788 (±0.1603) | alpha 2.18e-02 | 23/27 nonzero | 0.2s
Degree 3: CV MSE 1.0603 (±0.0559) | alpha 9.60e-03 | 66/83 nonzero | 0.2s
Degree 4: CV MSE 0.6122 (±0.0426) | alpha 7.60e-03 | 124/209 nonzero | 0.2s
Degree 5: CV MSE 0.3403 (±0.0247) | alpha 9.60e-03 | 135/461 nonzero | 0.5s
Degree 6: CV MSE 0.3563 (±0.0269) | alpha 1.08e-02 | 151/923 nonzero | 26.4s
Degree 7: CV MSE 0.3744 (±0.0276) | alpha 1.21e-02 | 173/1715 nonzero | 82.0s

Sweep finished in 109.5s

 degree  nonzero   alpha  cv_mse      se
      2       23 0.02179 2.97879 0.16028
      3       66 0.00960 1.06028 0.05586
      4      124 0.00760 0.61225 0.04264
      5      135 0.00960 0.34026 0.02467
      6      151 0.01079 0.35627 0.02694
      7      173 0.01213 0.37442 0.02760

Lowest CV MSE : degree 5 -> 0.3403 ± 0.0247
1-SE choice   : degree 5 -> 0.3403
CV R² (chosen): 0.9678
Test (1-SE choice, degree 5): MSE 0.2884 | R² 0.9736
Test (Lowest CV, degree 5): MSE 0.2884 | R² 0.9736

Degree 5: 135 of 461 te

In [22]:
import numpy as np
import pandas as pd
import torch

DEG, ALPHA, L1_RATIO, EPOCHS = 5, 0.005, 0.99, 750   # same settings as Stage 2

# ---- Train on all 1000 training rows ----
torch.manual_seed(42)
X_train_poly = get_polynomial_features_torch_with_interactions(X_raw_torch, DEG)
beta = elastic_net_adam_torch(X_train_poly, Y_torch, l1_ratio=L1_RATIO,
                              alpha=ALPHA, lr=0.05, epochs=EPOCHS)

with torch.no_grad():
    train_mse = torch.mean((Y_torch - X_train_poly @ beta) ** 2).item()
print(f"Training MSE: {train_mse:.4f}  (CV estimate was 0.3148)")

# ---- Predict the test set ----
X_test_raw = df_test1.iloc[:, :6].values
X_test_torch = torch.tensor(X_test_raw, dtype=torch.float64, device=device)
X_test_poly = get_polynomial_features_torch_with_interactions(X_test_torch, DEG)
with torch.no_grad():
    y_pred = (X_test_poly @ beta).cpu().numpy().ravel()

# ---- Sanity checks ----
y_train = Y_torch.cpu().numpy().ravel()
print(f"Test rows: {len(y_pred)} | NaNs: {np.isnan(y_pred).sum()}")
print(f"Train y range: [{y_train.min():.2f}, {y_train.max():.2f}] | "
      f"Pred range: [{y_pred.min():.2f}, {y_pred.max():.2f}]")
print(f"Test inputs within [-1, 1]: {np.all(np.abs(X_test_raw) <= 1 + 1e-9)}")

# ---- Save ----
out = df_test1.copy()
out['y'] = y_pred
out.to_csv("BT2024213_pred_var1.csv", index=False)
print("Saved BT2024213_pred_var1.csv")
display(out.head())

Training MSE: 0.2260  (CV estimate was 0.3148)
Test rows: 1000 | NaNs: 0
Train y range: [-10.31, 14.39] | Pred range: [-9.79, 15.74]
Test inputs within [-1, 1]: True
Saved BT2024213_pred_var1.csv


,x1,x2,x3,x4,x5,x6,y
0,1.000000,0.468421,1.000000,1.000000,1.000000,-1.000000,-4.110681
1,-0.478846,-0.775865,-0.426413,-1.000000,0.232556,-0.288742,-0.376065
2,-1.000000,1.000000,-1.000000,0.832330,-0.503098,-1.000000,7.840615
3,1.000000,0.903710,1.000000,0.948196,1.000000,-0.194109,-6.030154
4,1.000000,1.000000,0.625929,0.780418,0.933867,-1.000000,-0.941802
